<a href="https://colab.research.google.com/github/matthewpecsok/IS4490_student_course_files/blob/main/module-05-assignment-05-prompt-tool-workflow-template.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<a href="https://colab.research.google.com/github/matthewpecsok/IS4490-creation-fall2026/blob/main/module-05-ai-augmented-workflows/module-05-assignment-05-prompt-tool-workflow-template.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 5 Assignment 5: Sales Meeting Prep Workflow

**Notebook:** Student Build Template
**Runtime:** Jupyter or Google Colab with Ollama and LangChain
**Model:** `gemma4:12b` through Ollama

A sales rep is about to meet with a customer and wants to research the account first. In this notebook you will build four tools that let an assistant answer that research request one question at a time:

1. Who is this customer? (`get_customer_profile`)
2. What have they purchased? (`get_purchase_history`)
3. Are there any open support issues? (`get_support_cases`)
4. Can you draft me a short brief for the meeting? (`write_meeting_brief`)

Each tool's result is remembered and fed into the next step, so the conversation feels continuous even though every model call is stateless underneath. That "memory" is plumbing the notebook provides for you. The four tools are what you build.

This is still not an agent. The **order** of the four steps is fixed by the notebook. The model only decides, at each fixed step, whether the rep's question requires calling that step's tool and with what arguments.


## Important Instructions

1. Read the Module 5 assignment before editing this notebook.
2. Complete the cells marked `TODO`.
3. Use only the fictional data created in this notebook.
4. Do not connect to live CRM, sales, finance, email, or customer-communication systems.
5. Leave visible evidence of your tool tests, chat steps, failures, revisions, and final test results.
6. The language model may draft or explain text. Deterministic code must handle field exclusion, no-match behavior, arithmetic, the validation checkpoint, and the disclaimer.
7. Several functions intentionally raise `NotImplementedError`. Replace those placeholders with your own code before running the checkpoints.


---
## Setup: Installing Ollama and LangChain

If you are using Google Colab, run the setup cells below. If you are running locally and already have Ollama installed and running, you may skip the install cells and start at the imports cell.

This notebook uses `gemma4:12b` because it supports tool calling through LangChain.

In [ ]:
# Colab setup step 1: install zstd, which the Ollama installer expects on Ubuntu.
!apt-get install -y zstd

In [ ]:
# Colab setup step 2: install Ollama.
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
# Colab setup step 3: start the Ollama server and pull the model.
import subprocess
import time

ollama_server = subprocess.Popen(["ollama", "serve"], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
time.sleep(5)

pull = subprocess.run(["ollama", "pull", "gemma4:12b"], capture_output=True, text=True)
print(pull.stdout[-1000:])
print(pull.stderr[-1000:])
print("Ollama setup attempted. If the pull failed, check the runtime log and rerun this cell.")

In [ ]:
# Colab setup step 4: install Python packages.
!pip install langchain langchain-ollama --quiet

In [ ]:
from datetime import datetime

from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.tools import tool

MODEL = "gemma4:12b"
llm = ChatOllama(model=MODEL, temperature=0)

print(f"Ready. Using {MODEL} through Ollama and LangChain.")

In [ ]:
def require_finished(label, value):
    """Stop before a run when required student work is unfinished."""
    if value is None or "TODO" in str(value):
        raise ValueError(f"Complete {label} before running this cell.")


def pretty(obj):
    import json
    print(json.dumps(obj, indent=2, sort_keys=True))


def money(value):
    return f"${float(value):,.2f}"

---
## Skill and Workflow Scenario: Sales Meeting Prep

A sales rep is meeting with a customer soon and opens an internal assistant to prepare. The **Sales Meeting Prep Skill** answers three research questions and then drafts a short brief.

The skill is the goal-directed capability: get a rep ready for a customer meeting using only verified internal records.

The workflow is the fixed sequence that implements it:

1. Look up the customer's account profile.
2. Look up the customer's purchase history.
3. Look up the customer's support case history.
4. Validate that all three lookups are complete, draft a short brief, append a human-review disclaimer, and save it to a file.

The skill prepares research and a draft brief only. It never updates the CRM, contacts the customer, or makes a business decision. A person must review the AI-drafted brief before using it.


---
## Build Map From Earlier Modules

| Earlier pattern | Where to reuse it in Module 5 |
|---|---|
| Module 2 prompt calls | Build `SystemMessage` and `HumanMessage`, call `llm.invoke`, inspect the response. |
| Module 3 repeatable runs | Keep run evidence, compare expected vs. actual results, revise and rerun after a failure. |
| Module 4 tool categories | Use `@tool`, strong docstrings, typed inputs, direct `.invoke()` tests, controlled no-match behavior. |

The new work in Module 5 is chaining: letting one tool's output become the next step's context, and adding one deterministic checkpoint before AI-drafted content is allowed to leave the notebook as a file.


---
## Skill Definition

Before building the tools, define the business skill your four tools implement together. This is a design artifact — Ollama does not create a native "skill" object for you.


In [ ]:
sales_meeting_prep_skill = {
    "skill_name": "Sales Meeting Prep Skill",
    "business_goal": "TODO: state the recurring business capability this skill provides",
    "trigger_or_input": "TODO: describe what starts this skill (a rep naming a customer before a meeting)",
    "final_output": "TODO: describe the saved meeting-brief file",
    "tools_used_in_order": [
        "TODO: get_customer_profile",
        "TODO: get_purchase_history",
        "TODO: get_support_cases",
        "TODO: write_meeting_brief"
    ],
    "what_the_model_may_decide": [
        "TODO: which tool to call at each fixed step, and with what arguments"
    ],
    "what_the_model_may_never_decide": [
        "the order of the four steps",
        "whether the disclaimer is included in the saved brief",
        "whether a brief may be produced from incomplete research"
    ],
    "human_responsibilities": [
        "TODO: review the brief before the meeting, verify facts, decide what to say to the customer"
    ],
    "prohibited_actions": [
        "update the CRM",
        "contact or email the customer",
        "use real customer or purchase data",
        "approve or complete any business transaction"
    ]
}

pretty(sales_meeting_prep_skill)


## Responsible AI and Company-Data Check

Complete this check before building the tools. It forces you to define what your tools are allowed to expose and what stays human responsibility.

In [ ]:
company_data_responsibility_check = {
    "data_source": "Fictional local SQLite tables created inside this notebook",
    "allowed_fields": [
        "TODO: list the fields your tools may return, table by table"
    ],
    "excluded_fields": [
        "internal_risk_notes (customers table) -- internal use only, must never be returned by get_customer_profile"
    ],
    "authorized_user": "TODO: identify who should be allowed to run this workflow",
    "purpose_limit": "TODO: state the business question this workflow supports",
    "human_review_trigger": "TODO: state what must happen before the brief is used in a real meeting",
    "logging_need": "TODO: state what should be recorded so this workflow can be audited later"
}

pretty(company_data_responsibility_check)

---
## Part 1: Fictional CRM Data

Three staged tables represent a small CRM: customer accounts, purchase orders, and support cases. Do not replace this with real CRM, sales, or support data.

Notice the `internal_risk_notes` column on the `customers` table. It is deliberately included so you can practice excluding a field your tool should never return.


In [ ]:
import sqlite3

conn = sqlite3.connect("module5_sales_prep.db")
cursor = conn.cursor()

cursor.execute("DROP TABLE IF EXISTS customers")
cursor.execute(
    '''
    CREATE TABLE customers (
        customer_id TEXT PRIMARY KEY,
        account_name TEXT,
        industry TEXT,
        region TEXT,
        account_owner TEXT,
        relationship_since TEXT,
        contract_status TEXT,
        internal_risk_notes TEXT
    )
    '''
)

cursor.execute("DROP TABLE IF EXISTS purchase_history")
cursor.execute(
    '''
    CREATE TABLE purchase_history (
        order_id TEXT PRIMARY KEY,
        customer_id TEXT,
        item_description TEXT,
        amount REAL,
        order_date TEXT,
        status TEXT
    )
    '''
)

cursor.execute("DROP TABLE IF EXISTS support_cases")
cursor.execute(
    '''
    CREATE TABLE support_cases (
        case_id TEXT PRIMARY KEY,
        customer_id TEXT,
        subject TEXT,
        status TEXT,
        priority TEXT,
        opened_date TEXT
    )
    '''
)

customers = [
    ("CUST-1001", "Precision Parts Co.", "Manufacturing", "Midwest", "Sarah Chen", "2019", "active",
     "No concerns noted."),
    ("CUST-1002", "TechFlow Systems", "Technology", "West", "Rachel Torres", "2021", "active",
     "Expansion opportunity -- has expressed interest in additional licenses."),
    ("CUST-1003", "NovaMed Devices", "Healthcare", "East", "Linda Patel", "2017", "active",
     "No concerns noted."),
    ("CUST-1004", "Heritage Foods", "Food and Beverage", "Midwest", "Mark Johnson", "2022", "at-risk",
     "Renewal at risk -- competitor engagement rumored. Do not share this note directly with the customer."),
    ("CUST-1005", "Coastal Logistics", "Logistics", "East", "David Park", "2020", "active",
     "No concerns noted."),
    ("CUST-1006", "Summit Steel Works", "Manufacturing", "South", "James Okafor", "2026", "active",
     "New account, onboarding in progress.")
]
cursor.executemany("INSERT INTO customers VALUES (?, ?, ?, ?, ?, ?, ?, ?)", customers)

purchase_orders = [
    ("O-3001", "CUST-1001", "Precision tooling components", 18500.00, "2025-02-14", "completed"),
    ("O-3002", "CUST-1001", "Annual maintenance contract", 9200.00, "2025-08-01", "completed"),
    ("O-3003", "CUST-1001", "Replacement parts order", 4300.00, "2026-05-20", "pending"),
    ("O-3011", "CUST-1002", "Cloud analytics platform license", 42000.00, "2025-01-10", "completed"),
    ("O-3012", "CUST-1002", "Professional services - implementation", 15800.00, "2025-04-22", "completed"),
    ("O-3021", "CUST-1003", "Diagnostic device units (x12)", 96000.00, "2024-11-05", "completed"),
    ("O-3022", "CUST-1003", "Extended warranty package", 8000.00, "2025-11-05", "completed"),
    ("O-3031", "CUST-1004", "Ingredient supply contract Q1", 5200.00, "2025-01-15", "completed"),
    ("O-3032", "CUST-1004", "Ingredient supply contract Q2", 5400.00, "2025-04-15", "cancelled"),
    ("O-3041", "CUST-1005", "Freight management software", 22000.00, "2025-06-01", "completed")
]
cursor.executemany("INSERT INTO purchase_history VALUES (?, ?, ?, ?, ?, ?)", purchase_orders)

support_cases = [
    ("SC-501", "CUST-1001", "Delayed shipment on replacement parts", "open", "medium", "2026-06-01"),
    ("SC-502", "CUST-1001", "Invoice discrepancy resolved", "closed", "low", "2025-09-10"),
    ("SC-511", "CUST-1002", "API rate limit errors", "open", "high", "2026-06-25"),
    ("SC-521", "CUST-1003", "Device calibration question", "closed", "low", "2025-12-01"),
    ("SC-531", "CUST-1004", "Late delivery complaint", "open", "high", "2025-05-01"),
    ("SC-532", "CUST-1004", "Billing dispute", "open", "medium", "2025-07-10")
]
cursor.executemany("INSERT INTO support_cases VALUES (?, ?, ?, ?, ?, ?)", support_cases)

conn.commit()

print("Loaded", len(customers), "customers,", len(purchase_orders), "purchase orders, and", len(support_cases), "support cases.")
print("Note: CUST-1006 has no purchase orders and no support cases yet -- a brand-new account.")
print("Note: CUST-1005 has purchases but no support cases at all.")

---
## A Small Helper: Chatbot Memory Across Steps

This is the plumbing behind the chatbot. It is provided for you so you can study it, not build it.

Every call to the local model is stateless -- the model does not remember earlier turns on its own. `ask_chatbot_step` sends one chat turn: it takes the rep's question, an optional block of context text, and the one tool available at this step. It prints what happened and returns a small dictionary describing it -- which tool was called, with what arguments, what the tool returned, and the model's final answer.

`ask_chatbot_step` does **not** save anything to `RESEARCH_TRAIL` by itself. That is deliberate. The saving happens in the notebook cell that calls it, one explicit line at a time, so the "memory" is something you can actually see happening in your own code instead of something hidden inside a helper function.

The pattern you will repeat at each step looks like this:

```python
step1 = ask_chatbot_step(question, [some_tool], context_text=format_research_trail())

RESEARCH_TRAIL["some_key"] = step1["tool_result"]   # <-- this line is the chaining
```

`format_research_trail()` is a second small helper. It just reads whatever is currently in `RESEARCH_TRAIL` and turns it into readable text so it can be dropped into the next step's system prompt. It does not write anything -- only your own `RESEARCH_TRAIL[...] = ...` lines do that.


In [ ]:
RESEARCH_TRAIL = {
    "customer_id": None,
    "profile": None,
    "purchase_history": None,
    "support_cases": None
}


def reset_research_trail():
    for key in RESEARCH_TRAIL:
        RESEARCH_TRAIL[key] = None


def format_research_trail():
    """Turn whatever is currently in RESEARCH_TRAIL into readable text.
    This only reads RESEARCH_TRAIL -- it never writes to it.
    """
    lines = [f"{key}: {value}" for key, value in RESEARCH_TRAIL.items() if value]
    return "\n".join(lines) if lines else "No research has been gathered yet this session."


def ask_chatbot_step(user_message, tools, context_text=None):
    """Send one chat turn to the model and let it decide whether to call the
    provided tool.

    This function does NOT save anything to RESEARCH_TRAIL. It just runs one
    turn and hands back what happened: which tool was called, with what
    arguments, what the tool returned, and the model's final answer. The cell
    that calls this function decides what is worth remembering and writes it
    into RESEARCH_TRAIL itself -- watch for that line in each step below.
    """
    system_prompt = (
        "You are an internal sales-prep assistant. Only call a tool when the "
        "sales rep's question actually requires it. Use the customer_id already "
        "on file below when the rep does not repeat it."
    )
    if context_text:
        system_prompt += f"\n\nResearch gathered so far:\n{context_text}"

    tool_map = {t.name: t for t in tools}
    llm_with_tools = llm.bind_tools(tools)

    messages = [SystemMessage(content=system_prompt), HumanMessage(content=user_message)]
    response = llm_with_tools.invoke(messages)

    print(f"Sales rep: {user_message}")

    if not response.tool_calls:
        print("Tool called: none")
        print(f"Assistant: {response.content}\n")
        return {"tool_called": None, "arguments": None, "tool_result": None, "final_answer": response.content}

    call = response.tool_calls[0]
    tool_obj = tool_map[call["name"]]
    result = tool_obj.invoke(call["args"])

    print(f"Tool called: {call['name']}")
    print(f"Arguments: {call['args']}")
    print(f"Tool result: {result}")

    from langchain_core.messages import ToolMessage
    messages.append(response)
    messages.append(ToolMessage(content=str(result), tool_call_id=call["id"], name=call["name"]))
    final = llm_with_tools.invoke(messages)
    print(f"Assistant: {final.content}\n")

    return {
        "tool_called": call["name"],
        "arguments": call["args"],
        "tool_result": result,
        "final_answer": final.content
    }

---
## Step 1: Customer Profile

**Business problem:** before anything else, the assistant needs to know who the customer is. This tool looks up one customer's account profile by `customer_id`.


In [ ]:
@tool
def get_customer_profile(customer_id: str) -> str:
    """TODO: Write a precise description. Should look up one customer's account
    profile by customer_id. Use this when the sales rep asks about a customer's
    account details, account owner, industry, region, or how long they have been
    a customer. Do not use this for purchase history, support cases, or to draft
    the meeting brief.
    """
    # TODO: Implement this tool.
    # Requirements:
    # 1. Normalize customer_id (strip whitespace, uppercase).
    # 2. Query the customers table for that customer_id.
    # 3. If there is no match, return a plain string saying no customer was found
    #    for that ID. Do not invent a profile.
    # 4. If found, return a short plain-text summary with account_name, industry,
    #    region, account_owner, relationship_since, and contract_status.
    # 5. Do NOT include internal_risk_notes in the returned string. That field is
    #    for internal use only and must never be surfaced through this tool.
    raise NotImplementedError("Complete get_customer_profile before running the workflow.")

### Direct Tool Tests

Run the tool directly before placing it inside a chat step.

In [ ]:
# TODO: After implementing the tool, run these direct tests and leave the outputs visible.
print(get_customer_profile.invoke({"customer_id": "CUST-1001"}))
print()
print(get_customer_profile.invoke({"customer_id": "CUST-9999"}))  # unknown customer -- should not invent a profile

### Ask the Chatbot: Step 1

The sales rep names the customer they are meeting. The assistant should call `get_customer_profile`.

In [ ]:
step1 = ask_chatbot_step(
    "Can you pull up the account profile for customer CUST-1001? I have a meeting with them next week.",
    [get_customer_profile],
    context_text=format_research_trail()
)

# This is the chaining step: explicitly save what we learned so Steps 2-4 can use it.
RESEARCH_TRAIL["customer_id"] = step1["arguments"]["customer_id"]
RESEARCH_TRAIL["profile"] = step1["tool_result"]

print("RESEARCH_TRAIL now contains:")
pretty(RESEARCH_TRAIL)

---
## Step 2: Purchase History

**Business problem:** the rep needs to know the buying relationship before walking into the meeting. This tool looks up a customer's purchase orders and calculates the total lifetime purchase value.


In [ ]:
@tool
def get_purchase_history(customer_id: str) -> str:
    """TODO: Write a precise description. Should look up a customer's purchase
    orders by customer_id, most recent first, and report the total lifetime
    purchase value. Use this when the sales rep asks what a customer has bought
    or ordered. Do not use this for account profile or support case information.
    """
    # TODO: Implement this tool.
    # Requirements:
    # 1. Normalize customer_id.
    # 2. Query purchase_history for that customer_id, most recent order_date first.
    # 3. If there are no orders, return a clear "no purchase history found" string.
    #    Do not invent orders.
    # 4. Otherwise return a plain-text list of orders (item, amount, date, status)
    #    plus a total dollar amount across all orders.
    # 5. Calculate the total in code -- do not ask the model to add it up.
    raise NotImplementedError("Complete get_purchase_history before running the workflow.")

In [ ]:
# TODO: After implementing the tool, run these direct tests and leave the outputs visible.
print(get_purchase_history.invoke({"customer_id": "CUST-1001"}))
print()
print(get_purchase_history.invoke({"customer_id": "CUST-1006"}))  # brand-new account -- no purchases yet

### Ask the Chatbot: Step 2

Notice the rep does not repeat the customer ID. The assistant should reuse `RESEARCH_TRAIL["customer_id"]` from Step 1.

In [ ]:
step2 = ask_chatbot_step(
    "What has this customer purchased from us so far?",
    [get_purchase_history],
    context_text=format_research_trail()
)

RESEARCH_TRAIL["purchase_history"] = step2["tool_result"]

print("RESEARCH_TRAIL now contains:")
pretty(RESEARCH_TRAIL)

---
## Step 3: Support Cases

**Business problem:** the rep does not want to be surprised by an open complaint in the meeting. This tool looks up a customer's support case history and calculates how many cases are currently open.


In [ ]:
@tool
def get_support_cases(customer_id: str) -> str:
    """TODO: Write a precise description. Should look up a customer's support
    cases by customer_id and report how many are currently open. Use this when
    the sales rep asks about support issues, complaints, or open cases. Do not
    use this for purchase or profile data.
    """
    # TODO: Implement this tool.
    # Requirements:
    # 1. Normalize customer_id.
    # 2. Query support_cases for that customer_id.
    # 3. If there are no cases at all, return a clear "no support cases found"
    #    string. Do not invent problems.
    # 4. Otherwise list each case (subject, status, priority, opened_date) and
    #    report the count of cases where status == "open".
    # 5. Calculate the open-case count in code -- do not ask the model to count.
    raise NotImplementedError("Complete get_support_cases before running the workflow.")

In [ ]:
# TODO: After implementing the tool, run these direct tests and leave the outputs visible.
print(get_support_cases.invoke({"customer_id": "CUST-1001"}))
print()
print(get_support_cases.invoke({"customer_id": "CUST-1003"}))  # has case history, but zero currently open
print()
print(get_support_cases.invoke({"customer_id": "CUST-1006"}))  # no cases at all

### Ask the Chatbot: Step 3

In [ ]:
step3 = ask_chatbot_step(
    "Are there any support issues I should know about before the meeting?",
    [get_support_cases],
    context_text=format_research_trail()
)

RESEARCH_TRAIL["support_cases"] = step3["tool_result"]

print("RESEARCH_TRAIL now contains:")
pretty(RESEARCH_TRAIL)

---
## Step 4: Write the Meeting Brief

**Business problem:** the rep wants one short document to walk in with. This is the one place in the workflow where deterministic code, not the model, makes a decision: it checks that research is actually complete before letting the model draft anything, and it guarantees a human-review disclaimer is always present.

The disclaimer text below must appear in the saved file exactly as written, even if you change the drafting prompt later.


In [ ]:
BRIEF_DISCLAIMER = (
    "\n\n---\n"
    "This brief was drafted with AI assistance from internal records. "
    "Verify all facts before the meeting. Do not rely on this document without human review."
)

In [ ]:
@tool
def write_meeting_brief(customer_id: str) -> str:
    """TODO: Write a precise description. Should validate that profile,
    purchase-history, and support-case research is already complete for this
    customer_id, draft a short meeting-prep brief from that research, append
    the required disclaimer, and save the result to a text file. Use this only
    after the rep has already asked about profile, purchases, and support
    cases. Do not use this to look up new data or to approve or complete
    anything.
    """
    # TODO: Implement this tool.
    # Requirements:
    # 1. Normalize customer_id and compare it against RESEARCH_TRAIL["customer_id"].
    # 2. Validation checkpoint: if RESEARCH_TRAIL["customer_id"] does not match,
    #    or if profile / purchase_history / support_cases is missing, do NOT
    #    draft a report. Return a plain string naming which step is missing and
    #    asking the rep to complete it first.
    # 3. If all three are present, build a short prompt that gives the model the
    #    three research strings and asks for a brief, factual 2-4 paragraph
    #    meeting-prep summary. The model must not invent facts beyond what is in
    #    the three research strings, and must not claim any action was taken.
    # 4. Call the model once (llm.invoke) to draft the summary text.
    # 5. Append BRIEF_DISCLAIMER to the end of the drafted text. Do not let the
    #    model draft this sentence itself.
    # 6. Save the combined text to a file named f"meeting_brief_{customer_id}.txt".
    # 7. Return a short confirmation string naming the file and reminding the
    #    rep to review it before the meeting.
    raise NotImplementedError("Complete write_meeting_brief before running the workflow.")

### Direct Tool Tests

Test the validation checkpoint before you test the happy path.

In [ ]:
# TODO: After implementing the tool, run these direct tests and leave the outputs visible.

# Test A: call the report tool for a customer whose research is NOT complete yet.

print(write_meeting_brief.invoke({"customer_id": "CUST-1002"}))
print()

# Test B: complete the research trail for CUST-1001 (from Steps 1-3 above should
# already be in RESEARCH_TRAIL), then call the report tool for the happy path.
print(write_meeting_brief.invoke({"customer_id": "CUST-1001"}))
print()

# Test C: open the saved file and confirm the disclaimer is present.
with open("meeting_brief_CUST-1001.txt") as f:
    print(f.read())

### Ask the Chatbot: Step 4

In [ ]:
step4 = ask_chatbot_step(
    "Can you put together a short brief I can bring into the meeting?",
    [write_meeting_brief],
    context_text=format_research_trail()
)

print("Final answer to the rep:")
print(step4["final_answer"])

---
## Part 2: Required Tests

Run at least six tests. You may reuse the direct-tool-test evidence above and add the remaining cases here.

Required coverage:

1. Full valid run for one customer: profile, purchases, support cases, then a saved brief with the disclaimer present.
2. Unknown `customer_id` at the profile step -- no invented profile.
3. A customer with no purchase history -- no invented orders.
4. A customer with no open support cases -- correct count, no invented problems.
5. Calling `write_meeting_brief` before completing all three research steps -- blocked with a clear message.
6. Opening the saved `.txt` file and confirming the disclaimer text is present exactly as specified.

At least one test should reveal a real weakness -- a query bug, a description that leads the model to skip a tool, or a validation gap. Revise it, rerun the test, and keep the before-and-after evidence visible.


---
## Part 3: Reflection

Answer these questions in your submitted notebook.

1. What is the goal of the Sales Meeting Prep Skill, and what does each of the four steps contribute to it?
2. What did the model decide at each step, and what did the workflow decide for it instead?
3. What information passed from Step 1 into Steps 2, 3, and 4 so the assistant appeared to "remember" the conversation?
4. Why must the total purchase value and open-case count be calculated in code rather than by the model?
5. Why must the disclaimer be inserted by code instead of requested in the drafting prompt?
6. What would change if the model were allowed to decide the order of these four steps itself?
7. What data should never be exposed if this workflow were connected to a real CRM?
8. What did you use AI to help with, and what did you personally run, test, and verify?
